# Training Logic Testing

Interactive smoke tests for all current training regimes on the same real dummy `adata` source used by the other notebooks.

This notebook is for quick behavior checks (few batches), not benchmark training.

## Notes

- Uses `experiment=notebook_training_logic` to pin `data/dummy/manifest.csv` and `data/dummy/processed`.
- Runs each regime as a smoke test on CPU with small batch limits.
- You can increase training depth by overriding trainer limits in the scenario calls.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import hydra
import pandas as pd
import rootutils
import torch
from hydra.core.global_hydra import GlobalHydra
from hydra.core.hydra_config import HydraConfig
from omegaconf import DictConfig, OmegaConf, open_dict

rootutils.setup_root("..", indicator=".project-root", pythonpath=True)

from src.train import train

ROOT = Path(rootutils.find_root(indicator=".project-root"))
CONFIG_DIR = ROOT / "configs"
print(f"Project root: {ROOT}")
print(f"Config dir:   {CONFIG_DIR}")

Project root: /Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil
Config dir:   /Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/configs


In [2]:
def _to_float(value: Any) -> float | None:
    """Convert metric values (tensor/scalar) to Python floats when possible."""
    if value is None:
        return None
    if isinstance(value, torch.Tensor):
        if value.numel() == 0:
            return None
        return float(value.detach().cpu().flatten()[0].item())
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def run_training_scenario(
    name: str,
    task: str,
    model: str,
    extra_overrides: list[str] | None = None,
    print_cfg: bool = False,
) -> dict[str, Any]:
    """Compose Hydra config and run a short train+test cycle."""
    base_overrides = [
        "experiment=notebook_training_logic",
        "hydra/job_logging=stdout",
        "hydra/hydra_logging=none",
        "callbacks=none",
        "logger=null",
        f"model={model}",
        "trainer.accelerator=cpu",
        "trainer.devices=1",
        "trainer.max_epochs=1",
        "trainer.limit_train_batches=2",
        "trainer.limit_val_batches=1",
        "trainer.limit_test_batches=1",
        "trainer.log_every_n_steps=1",
    ]

    # BGRL task config includes a defaults-based scheduler override; some notebook
    # compose flows can fail resolving that override. In that case, inject task and
    # scheduler nodes explicitly after compose.
    if task != "pretrain_bgrl":
        base_overrides.append(f"task={task}")

    if extra_overrides:
        base_overrides.extend(extra_overrides)

    GlobalHydra.instance().clear()
    with hydra.initialize_config_dir(config_dir=str(CONFIG_DIR), version_base="1.3"):
        cfg: DictConfig = hydra.compose(
            config_name="train",
            overrides=base_overrides,
            return_hydra_config=True,
        )

    if task == "pretrain_bgrl":
        pretrain_task_cfg = OmegaConf.load(CONFIG_DIR / "task" / "pretrain_bgrl.yaml")
        step_sched_cfg = OmegaConf.load(CONFIG_DIR / "scheduler" / "cosine_step.yaml")
        with open_dict(cfg):
            cfg.task = pretrain_task_cfg.task
            cfg.scheduler = step_sched_cfg

    if print_cfg:
        print(OmegaConf.to_yaml(cfg))

    with open_dict(cfg):
        cfg.paths.root_dir = str(ROOT)
        cfg.extras.print_config = False
        # Keep optimizer/scheduler/task as config nodes for LightningModule init.
        cfg.model._recursive_ = False

    HydraConfig().set_config(cfg)
    metric_dict, _ = train(cfg)

    summary = {
        "scenario": name,
        "task": task,
        "model": model,
        "train/loss": _to_float(metric_dict.get("train/loss")),
        "val/loss": _to_float(metric_dict.get("val/loss")),
        "test/loss": _to_float(metric_dict.get("test/loss")),
        "train/acc": _to_float(metric_dict.get("train/acc")),
        "val/acc": _to_float(metric_dict.get("val/acc")),
        "test/acc": _to_float(metric_dict.get("test/acc")),
    }
    print(
        f"[{name}] train/loss={summary['train/loss']} val/loss={summary['val/loss']} test/loss={summary['test/loss']}"
    )
    return summary

## Data And Config Sanity Check

This confirms the notebook uses the same dummy manifest/adata pipeline as the other notebooks.

In [3]:
GlobalHydra.instance().clear()
with hydra.initialize_config_dir(config_dir=str(CONFIG_DIR), version_base="1.3"):
    cfg_preview = hydra.compose(
        config_name="train",
        overrides=[
            "experiment=notebook_training_logic",
            "task=finetune_mean",
            "model=supervised_mean_module",
            "hydra/job_logging=stdout",
            "hydra/hydra_logging=none",
        ],
        return_hydra_config=True,
    )

print("Manifest:", cfg_preview.data.raw_manifest_path)
print("Processed:", cfg_preview.data.processed_dir)
print("Adata expected via manifest:", ROOT / "data" / "dummy" / "imc_data.h5ad")
print(OmegaConf.to_yaml(cfg_preview.task))

Manifest: /Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/data//dummy/manifest.csv
Processed: /Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/data//dummy/processed
Adata expected via manifest: /Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/data/dummy/imc_data.h5ad
name: finetune_mean
aggregation: mean
bag_key: region_id
bag_fallback_key: sample_id
max_instances_per_bag: 0
instance_sampling: all
target_type: binary
target_columns: null
loss: categorical_bce



In [4]:
dm = hydra.utils.instantiate(cfg_preview.data)
dm.prepare_data()
dm.setup()
train_ds = dm.train_dataloader().dataset
example = train_ds[0]

print("Train dataset size:", len(train_ds))
print("Example x shape:", tuple(example.x.shape))
print("Example edge_index shape:", tuple(example.edge_index.shape))
if hasattr(example, "region_id"):
    print("Example region_id:", example.region_id)
if hasattr(example, "sample_id"):
    print("Example sample_id:", example.sample_id)

Train dataset size: 9
Example x shape: (762, 34)
Example edge_index shape: (2, 4536)
Example region_id: r1
Example sample_id: s1


/Users/lovrorabuzin/Projects/grass-mil_unification/grass-mil/src/data/components/datasets.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(entry.path)


## Run Training Regimes (Smoke)

Each cell runs one scenario and appends to `RESULTS`.

In [5]:
RESULTS: list[dict[str, Any]] = []

In [6]:
# 1) BGRL pretraining
RESULTS.append(
    run_training_scenario(
        name="bgrl_pretrain",
        task="pretrain_bgrl",
        model="bgrl_module",
    )
)

ValueError: Config group override must be a string or a list. Got NoneType

In [ ]:
# 2) Supervised mean
RESULTS.append(
    run_training_scenario(
        name="supervised_mean_default",
        task="finetune_mean",
        model="supervised_mean_module",
    )
)

NameError: name 'RESULTS' is not defined

In [ ]:
# 3) Supervised MIL default
RESULTS.append(
    run_training_scenario(
        name="supervised_mil_default",
        task="finetune_mil",
        model="supervised_mil_module",
    )
)

In [ ]:
# 4) MIL with region accumulation (hyperbatch threshold)
RESULTS.append(
    run_training_scenario(
        name="supervised_mil_region_accum",
        task="finetune_mil",
        model="supervised_mil_module",
        extra_overrides=[
            "task.region_accumulation.enabled=true",
            "task.region_accumulation.hyperbatch_size=2",
            "task.region_accumulation.flush_on_epoch_end=true",
        ],
    )
)

In [ ]:
# 5) MIL with node auxiliary loss (graph_label targets)
RESULTS.append(
    run_training_scenario(
        name="supervised_mil_node_aux_graph_label",
        task="finetune_mil",
        model="supervised_mil_module",
        extra_overrides=[
            "task.node_aux.enabled=true",
            "task.node_aux.target_mode=graph_label",
            "task.node_aux.loss_mode=bce",
            "task.node_aux.weight=0.2",
        ],
    )
)

In [ ]:
# 6) MIL with node auxiliary loss (attention_shaped_ti targets)
RESULTS.append(
    run_training_scenario(
        name="supervised_mil_node_aux_attention_shaped_ti",
        task="finetune_mil",
        model="supervised_mil_module",
        extra_overrides=[
            "task.node_aux.enabled=true",
            "task.node_aux.target_mode=attention_shaped_ti",
            "task.node_aux.loss_mode=bce",
            "task.node_aux.weight=0.2",
        ],
    )
)

In [ ]:
# 7) MIL with entropy regularization
RESULTS.append(
    run_training_scenario(
        name="supervised_mil_entropy_reg",
        task="finetune_mil",
        model="supervised_mil_module",
        extra_overrides=[
            "task.entropy_reg.enabled=true",
            "task.entropy_reg.mode=attention",
            "task.entropy_reg.weight=0.01",
        ],
    )
)

## Summary And Validation

In [ ]:
summary_df = pd.DataFrame(RESULTS)
summary_df

In [ ]:
loss_keys = ["train/loss", "val/loss", "test/loss"]

for row in RESULTS:
    scenario = row["scenario"]
    train_loss = row.get("train/loss")
    if train_loss is None:
        raise AssertionError(f"{scenario}: missing train/loss")
    if not torch.isfinite(torch.tensor(train_loss)):
        raise AssertionError(f"{scenario}: non-finite train/loss={train_loss}")

    for key in loss_keys[1:]:
        value = row.get(key)
        if value is None:
            continue
        if not torch.isfinite(torch.tensor(value)):
            raise AssertionError(f"{scenario}: non-finite {key}={value}")

print("All available losses are finite (train/loss required).")

## Optional: Extend Scenarios

- Increase depth by passing extra overrides like `trainer.max_epochs=3` or larger `trainer.limit_train_batches`.
- Toggle MIL options in combinations by reusing `run_training_scenario(..., extra_overrides=[...])`.
- Set `print_cfg=True` in `run_training_scenario` to inspect the fully composed config for a scenario.